In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [15]:
train = pd.read_csv('train.csv').drop('id', axis=1)
test = pd.read_csv('test.csv').drop('id', axis=1)
sample_submission = pd.read_csv('sample_submission.csv')

In [16]:
for df in [train, test]:

    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'], inplace=True)

    # df['MCSquared'] = df['MonthlyCharges'] ** 2
    # df['TCSquared'] = df['TotalCharges'] ** 2

    df['MCLog'] = np.log1p(df['MonthlyCharges'])
    df['TCLog'] = np.log1p(df['TotalCharges'])

    # df['MCSqrt'] = np.sqrt(df['MonthlyCharges'])
    # df['TCSqrt'] = np.sqrt(df['TotalCharges'])

    df['MCSqueezed'] = 1 / (df['MonthlyCharges'] + 1)
    df['TCSqueezed'] = 1 / (df['TotalCharges'] + 1)

    # Loyalty
    df["TenureLog"] = np.log1p(df["tenure"])
    # df["TenureSquared"] = df["tenure"]**2
    # df['TenureSqrt'] = np.sqrt(df['tenure'])
    df['TenureSqueezed'] = 1 / (df['tenure'] + 1)

    # Spending
    df["AvgMonthlySpend"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["SpendDeviation"] = df["MonthlyCharges"] - df["AvgMonthlySpend"]

    # Contract
    df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)

    # Payment
    df["IsElectronicCheck"] = (df["PaymentMethod"] == "Electronic check").astype(int)
    df["AutoPayment"] = df["PaymentMethod"].str.contains("automatic").astype(int)

    # Service Engagement
    df["ServiceCount"] = (
        (df["OnlineSecurity"] == "Yes").astype(int) +
        (df["OnlineBackup"] == "Yes").astype(int) +
        (df["DeviceProtection"] == "Yes").astype(int) +
        (df["TechSupport"] == "Yes").astype(int) +
        (df["StreamingTV"] == "Yes").astype(int) +
        (df["StreamingMovies"] == "Yes").astype(int)
    )

    # Internet
    df["IsFiber"] = (df["InternetService"] == "Fiber optic").astype(int)

    # Early Risk
    df["EarlyTenure"] = (df["tenure"] < 12).astype(int)
    df["HighChargeEarly"] = (
        (df["tenure"] < 12) &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    # Value
    df["LifetimeValue"] = df["MonthlyCharges"] * df["tenure"]

train['MCGT75%'] = train['MonthlyCharges'].gt(train['MonthlyCharges'].quantile(0.75)).astype(int)
test['MCGT75%'] = test['MonthlyCharges'].gt(train['MonthlyCharges'].quantile(0.75)).astype(int)

train['TCGT75%'] = train['TotalCharges'].gt(train['TotalCharges'].quantile(0.75)).astype(int)
test['TCGT75%'] = test['TotalCharges'].gt(train['TotalCharges'].quantile(0.75)).astype(int)

/var/folders/sk/bf5_j5395pn851dn2wx13dy80000gn/T/ipykernel_20897/4109498660.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'], inplace=True)
/var/folders/sk/bf5_j5395pn851dn2wx13dy80000gn/T/ipykernel_20897/4109498660.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on whic

In [17]:
X, y = train.drop('Churn', axis=1), train['Churn']
y = y.map({'Yes': 1, 'No': 0})

In [18]:
X.head(3)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,IsMonthToMonth,IsElectronicCheck,AutoPayment,ServiceCount,IsFiber,EarlyTenure,HighChargeEarly,LifetimeValue,MCGT75%,TCGT75%
0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,...,0,0,0,3,0,0,0,1742.9,0,0
1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,...,0,0,1,4,0,0,0,4031.0,0,0
2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,...,1,1,0,3,1,0,0,5823.2,1,1


In [19]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cat_features = X.select_dtypes(include='object').columns.tolist()
cat_features.append('SeniorCitizen')

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):

    print(f'Fold {fold+1}')

    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.05,
        depth=6,
        eval_metric='AUC',
        random_seed=42,
        verbose=200,
        early_stopping_rounds=200
    )

    model.fit(
        X_train, y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )

    oof_preds[valid_idx] = model.predict_proba(X_valid)[:, 1]
    test_preds += model.predict_proba(test)[:, 1] / skf.n_splits

Fold 1
0:	test: 0.8909342	best: 0.8909342 (0)	total: 720ms	remaining: 36m
200:	test: 0.9139735	best: 0.9139735 (200)	total: 48.9s	remaining: 11m 20s
400:	test: 0.9149977	best: 0.9149977 (400)	total: 1m 32s	remaining: 10m 1s
600:	test: 0.9154340	best: 0.9154345 (599)	total: 2m 14s	remaining: 8m 58s
800:	test: 0.9156952	best: 0.9156952 (800)	total: 2m 59s	remaining: 8m 12s
1000:	test: 0.9158526	best: 0.9158526 (1000)	total: 3m 40s	remaining: 7m 21s
1200:	test: 0.9159377	best: 0.9159384 (1196)	total: 4m 21s	remaining: 6m 31s
1400:	test: 0.9160315	best: 0.9160315 (1400)	total: 5m 2s	remaining: 5m 45s
1600:	test: 0.9160779	best: 0.9160809 (1565)	total: 5m 46s	remaining: 5m 2s
1800:	test: 0.9161180	best: 0.9161180 (1800)	total: 6m 27s	remaining: 4m 17s
2000:	test: 0.9161314	best: 0.9161335 (1920)	total: 7m 7s	remaining: 3m 33s
2200:	test: 0.9161441	best: 0.9161448 (2131)	total: 7m 47s	remaining: 2m 49s
2400:	test: 0.9161485	best: 0.9161639 (2220)	total: 8m 30s	remaining: 2m 7s
Stopped by ove

In [23]:
from sklearn.metrics import roc_auc_score

cv_score = roc_auc_score(y, oof_preds)
print("CV ROC-AUC:", cv_score)

CV ROC-AUC: 0.916533258194084


In [21]:
model_path = "models/catboost_model_fe.cbm"
model.save_model(model_path)

In [24]:
submission = sample_submission.copy()
submission['Churn'] = test_preds
submission.to_csv('submission.csv', index=False)